# Domain-Specific Question Answering with RAG Evaluation
### Prototyping & Pipeline Exploration Notebook
This notebook demonstrates each stage of the RAG pipeline step-by-step:
1. Document Ingestion & Cleaning
2. Recursive Chunking & Metadata Attachment
3. Embedding Generation (all-MiniLM-L6-v2) & Vector Store Indexing (ChromaDB)
4. Semantic Similarity Search Retrieval
5. Grounded Prompt Construction & LLM Inference
6. RAGAS Quality Metrics Calculation

In [ ]:
import os
import sys

# Ensure project root is in sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.ingestion.loader import DocumentLoader
from src.ingestion.chunker import DocumentChunker
from src.embeddings.embed import EmbeddingService, VectorStoreService
from src.retrieval.retriever import Retriever
from src.generation.prompt_builder import PromptBuilder
from src.generation.llm_service import LLMService
from src.evaluation.metrics import RAGEvaluatorMetrics
print("All modules imported successfully!")

## 1. Document Loading & Text Cleaning

In [ ]:
raw_dir = "../data/raw"
loader = DocumentLoader(raw_data_dir=raw_dir)
documents = loader.load_documents()

print(f"Loaded {len(documents)} document sections.")
if documents:
    print("Sample snippet:", documents[0]['text'][:300])

## 2. Chunking & Metadata Attachment

In [ ]:
chunker = DocumentChunker(chunk_size=400, chunk_overlap=50, processed_data_dir="../data/processed")
chunks = chunker.chunk_documents(documents)

print(f"Created {len(chunks)} chunks.")
if chunks:
    print("First chunk ID:", chunks[0]['chunk_id'])
    print("Metadata:", chunks[0]['metadata'])
    print("Text:", chunks[0]['text'][:200])

## 3. Embedding Generation & ChromaDB Indexing

In [ ]:
embed_service = EmbeddingService()
vectorstore = VectorStoreService(persist_dir="../vectorstore/chroma_db", embedding_service=embed_service)

vectorstore.reset_index()
indexed = vectorstore.index_chunks(chunks)
print(f"Indexed {indexed} chunks into ChromaDB.")

## 4. Semantic Similarity Search Retrieval

In [ ]:
retriever = Retriever(vectorstore_service=vectorstore, top_k=3)
test_query = "What is the minimum attendance required for exam eligibility?"
retrieved_chunks = retriever.retrieve(test_query)

for i, chunk in enumerate(retrieved_chunks):
    print(f"\n--- Chunk {i+1} (Score: {chunk['score']}) ---")
    print(f"Source: {chunk['source_document']} | Page: {chunk['page_number']}")
    print(chunk['text'][:150] + "...")

## 5. Grounded Prompt Construction & LLM Answer Generation

In [ ]:
prompt = PromptBuilder.build_prompt(test_query, retrieved_chunks)
print("Constructed Grounded Prompt:\n")
print(prompt[:600] + "...\n")

llm = LLMService()
response = llm.generate_answer(test_query, retrieved_chunks)
print("Generated Answer:\n", response['answer'])
print("\nLatency:", response['latency_ms'], "ms")

## 6. RAGAS Metrics Evaluation

In [ ]:
evaluator = RAGEvaluatorMetrics()
reference_answer = "A minimum aggregate attendance of 75% in each registered course is mandatory to appear for exams."

scores = evaluator.evaluate_sample(
    question=test_query,
    retrieved_contexts=[c['text'] for c in retrieved_chunks],
    generated_answer=response['answer'],
    reference_answer=reference_answer
)

import pandas as pd
df_scores = pd.DataFrame([scores])
df_scores